# Inclinação Política com BERTimbau + JANELAMENTO (long-document)

**Atividade Prática — Deep Learning para Demandas Reais**

Versão avançada da frente política. Em vez de **truncar** o discurso em 256 tokens (que perde o
sinal ideológico difuso), aqui usamos **janelamento**: cada discurso é fatiado em janelas de 256
tokens com sobreposição, o BERTimbau classifica cada janela, e as probabilidades são **agregadas
por média** para dar a predição do discurso inteiro.

Hipótese: vendo o documento completo, o BERTimbau deve **alcançar** os modelos de texto completo
(TF-IDF), que na versão truncada o superavam (0,77–0,78 vs. 0,73).

> `Ambiente de execução → GPU`, depois `Executar tudo`. Coleta + treino levam alguns minutos numa T4.


## 1. Dependências e GPU

In [ ]:
!pip install -q -U transformers accelerate scikit-learn
import torch
print("GPU:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Coletar discursos da Câmara (API pública)

In [ ]:
import requests, time, random
API="https://dadosabertos.camara.leg.br/api/v2"; H={"Accept":"application/json"}
LEGISLATURA=57; DATA_INI,DATA_FIM="2023-02-01","2024-12-31"
MIN_PALAVRAS=40; MAX_POR_DEP=12; ALVO=3000
MAPA={"PT":"esquerda","PSOL":"esquerda","PCDOB":"esquerda","PSB":"esquerda","PDT":"esquerda","REDE":"esquerda",
 "PL":"direita","PP":"direita","REPUBLICANOS":"direita","UNIÃO":"direita","NOVO":"direita","PSC":"direita","PSDB":"direita","PSL":"direita","PRTB":"direita"}
def get(u):
    for i in range(4):
        try: return requests.get(u,headers=H,timeout=30).json()
        except Exception: time.sleep(1.5*(i+1))
    return {"dados":[],"links":[]}
deps=[]; url=f"{API}/deputados?idLegislatura={LEGISLATURA}&ordem=ASC&ordenarPor=nome&itens=100"
while url:
    d=get(url); deps+=d.get("dados",[]); url=next((l["href"] for l in d.get("links",[]) if l["rel"]=="next"),None)
random.seed(42); random.shuffle(deps); linhas=[]
for dep in deps:
    part=(dep.get("siglaPartido") or "").strip().upper(); ideo=MAPA.get(part)
    if not ideo: continue
    n=0; u=f"{API}/deputados/{dep['id']}/discursos?itens=50&ordenarPor=dataHoraInicio&ordem=DESC&dataInicio={DATA_INI}&dataFim={DATA_FIM}"
    while u and n<MAX_POR_DEP:
        d=get(u)
        for disc in d.get("dados",[]):
            txt=(disc.get("transcricao") or "").strip()
            if len(txt.split())>=MIN_PALAVRAS:
                linhas.append({"texto":txt,"partido":part,"ideologia":ideo,"id_deputado":dep["id"]}); n+=1
                if n>=MAX_POR_DEP: break
        u=next((l["href"] for l in d.get("links",[]) if l["rel"]=="next"),None)
    if len(linhas)>=ALVO: break
import pandas as pd
dpol=pd.DataFrame(linhas); print("coletados:",len(dpol)); print(dpol["ideologia"].value_counts())

## 2b. Limpeza CRÍTICA — remover cabeçalho e siglas (evita vazamento)

In [ ]:
import re
SIG=r'\b(PT|PL|PP|PSOL|PCdoB|PCDOB|PSB|PDT|REDE|REPUBLICANOS|UNIÃO|UNIAO|NOVO|PSC|PSDB|PSL|PRTB|MDB|PSD)\b'
def limpar(t):
    t=re.sub(r'^.*?\)\s*-\s*','',str(t),count=1)
    return re.sub(SIG,'',t,flags=re.I)
dpol['texto']=dpol['texto'].map(limpar)
print(dpol['texto'].iloc[0][:200])

## 3. Rótulo, balanceamento e split por deputado

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
dpol["label"]=(dpol["ideologia"]=="direita").astype(int)
nmin=dpol["label"].value_counts().min()
dpol=pd.concat([g.sample(nmin,random_state=42) for _,g in dpol.groupby("label")]).reset_index(drop=True)
gss=GroupShuffleSplit(1,test_size=0.2,random_state=42); tr_i,te_i=next(gss.split(dpol,dpol["label"],groups=dpol["id_deputado"]))
tr,te=dpol.iloc[tr_i],dpol.iloc[te_i]
gss2=GroupShuffleSplit(1,test_size=0.15,random_state=42); tr2_i,va_i=next(gss2.split(tr,tr["label"],groups=tr["id_deputado"]))
va=tr.iloc[va_i]; tr=tr.iloc[tr2_i]
print("treino/val/teste (documentos):",len(tr),len(va),len(te))

## 4. Janelamento — fatiar cada discurso em janelas de 256 tokens

Cada discurso vira uma lista de janelas com sobreposição (stride). No **treino**, cada janela herda
o rótulo do discurso. Na **avaliação**, as janelas de um mesmo discurso são agregadas por média das
probabilidades, dando a predição em nível de documento.

In [ ]:
from transformers import AutoTokenizer
MODELO="neuralmind/bert-base-portuguese-cased"; MAX_LEN=256; STRIDE=64
tok=AutoTokenizer.from_pretrained(MODELO)
CLS,SEP,PAD=tok.cls_token_id,tok.sep_token_id,tok.pad_token_id
CORPO=MAX_LEN-2

def janelas(ids):
    if len(ids)<=CORPO: return [ids]
    out=[]; i=0
    while i<len(ids):
        out.append(ids[i:i+CORPO])
        if i+CORPO>=len(ids): break
        i+=CORPO-STRIDE
    return out

def montar(ids_janela):
    seq=[CLS]+ids_janela+[SEP]; attn=[1]*len(seq)
    while len(seq)<MAX_LEN: seq.append(PAD); attn.append(0)
    return seq,attn

class JanelasDS(torch.utils.data.Dataset):
    # expande cada documento em janelas (treino/val)
    def __init__(self, frame):
        self.itens=[]
        for txt,lab in zip(frame["texto"].astype(str), frame["label"]):
            ids=tok(txt, add_special_tokens=False)["input_ids"]
            for w in janelas(ids):
                s,a=montar(w); self.itens.append((s,a,int(lab)))
    def __len__(self): return len(self.itens)
    def __getitem__(self,i):
        s,a,lab=self.itens[i]
        return {"input_ids":torch.tensor(s),"attention_mask":torch.tensor(a),"labels":torch.tensor(lab)}

ds_tr, ds_va = JanelasDS(tr), JanelasDS(va)
print("janelas de treino:",len(ds_tr),"| janelas de val:",len(ds_va))

## 5. Modelo, perda ponderada e treino (em nível de janela)

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score
ytr=tr["label"].values; n=len(ytr); n1=int(ytr.sum()); n0=n-n1
pesos=torch.tensor([n/(2*max(n0,1)), n/(2*max(n1,1))], dtype=torch.float)
def metrics_janela(p):
    logits,labels=p; preds=np.argmax(logits,axis=-1)
    return {"f1_janela":f1_score(labels,preds,average="macro",zero_division=0),
            "acc_janela":accuracy_score(labels,preds)}
class TrainerP(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels=inputs.pop("labels"); out=model(**inputs)
        loss=torch.nn.CrossEntropyLoss(weight=pesos.to(out.logits.device))(out.logits,labels)
        return (loss,out) if return_outputs else loss
model=AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)
common=dict(output_dir="bertimbau_jan", learning_rate=2e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=3, weight_decay=0.01, warmup_ratio=0.1,
    load_best_model_at_end=True, metric_for_best_model="f1_janela", greater_is_better=True,
    logging_steps=25, report_to="none", fp16=torch.cuda.is_available(), seed=42)
try: args=TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **common)
except TypeError: args=TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **common)
trainer=TrainerP(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va, compute_metrics=metrics_janela)
trainer.train()

## 6. Avaliação em NÍVEL DE DOCUMENTO (agregando as janelas por média)

In [ ]:
import json
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
model.eval()
@torch.no_grad()
def prob_doc(texto):
    ids=tok(str(texto), add_special_tokens=False)["input_ids"]
    ps=[]
    for w in janelas(ids):
        s,a=montar(w)
        b={"input_ids":torch.tensor([s]).to(model.device),"attention_mask":torch.tensor([a]).to(model.device)}
        ps.append(torch.softmax(model(**b).logits,dim=-1)[0,1].item())
    return float(np.mean(ps))

probs=np.array([prob_doc(t) for t in te["texto"]])
yhat=(probs>=0.5).astype(int); yte=te["label"].values
print(classification_report(yte,yhat,target_names=["esquerda","direita"],digits=4))
macro=f1_score(yte,yhat,average="macro"); auc=roc_auc_score(yte,probs); cm=confusion_matrix(yte,yhat)
print(f"[DOCUMENTO] macro-F1={macro:.4f} | ROC-AUC={auc:.4f}")
json.dump({"modelo":MODELO+" + janelamento","macro_f1":round(float(macro),4),"roc_auc":round(float(auc),4),"cm":cm.tolist()},
          open("resultados_bertimbau_janelamento.json","w"),ensure_ascii=False,indent=2)

## 7. Gráficos

In [ ]:
import matplotlib.pyplot as plt
h=trainer.state.log_history
tl=[(x["epoch"],x["loss"]) for x in h if "loss" in x and "epoch" in x]
el=[(x["epoch"],x["eval_loss"]) for x in h if "eval_loss" in x]
plt.figure(figsize=(6,4))
if tl: plt.plot(*zip(*tl),label="loss treino",color="#2f6fd0")
if el: plt.plot(*zip(*el),"o-",label="loss validação",color="#0f9d72")
plt.xlabel("Época"); plt.ylabel("Loss"); plt.title("Loss — BERTimbau + janelamento"); plt.legend(); plt.tight_layout()
plt.savefig("fig_loss_bertimbau_jan.png",dpi=150); plt.show()
plt.figure(figsize=(4.4,3.8)); plt.imshow(cm,cmap="Blues")
for i in range(2):
    for j in range(2): plt.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black",fontsize=13)
plt.xticks([0,1],["esquerda","direita"]); plt.yticks([0,1],["esquerda","direita"])
plt.xlabel("Predito"); plt.ylabel("Real"); plt.title("Confusão — BERTimbau + janelamento (documento)")
plt.colorbar(); plt.tight_layout(); plt.savefig("fig_matriz_confusao_bertimbau_jan.png",dpi=150); plt.show()

## 8. Testar com texto novo e salvar

In [ ]:
def prever(texto):
    p=prob_doc(texto); rot="DIREITA" if p>=0.5 else "ESQUERDA"
    return f"{rot} (confiança {max(p,1-p)*100:.1f}%)"
print(prever("Defendo a redução de impostos e a liberdade econômica do empreendedor."))
print(prever("É dever do Estado garantir saúde pública universal e reduzir a desigualdade."))
trainer.save_model("modelo_bertimbau_janelamento"); tok.save_pretrained("modelo_bertimbau_janelamento")
try:
    from google.colab import files
    for fn in ["resultados_bertimbau_janelamento.json","fig_loss_bertimbau_jan.png","fig_matriz_confusao_bertimbau_jan.png"]:
        files.download(fn)
except Exception as e: print("Baixe pelos Arquivos.", e)